# PySpark for Data Engineers — Phase 0 & 1

---

## Phase 0 — Setup & Mental Model
**Duration:** Week 1 (\~3 days) | **Goal:** Understand *why* Spark exists and get your first code running.

By the end of Phase 0 you will be able to explain in one sentence:
> "Transformations are lazy — nothing runs until an action is called."

---

## 0.1 — The Big-Data Problem

**Why does single-machine processing break?**

Imagine you have a 500 GB CSV of taxi trips. On a single machine:

1. **Memory wall** — pandas loads the entire file into RAM. If your laptop has 16 GB, the program crashes immediately (`MemoryError`).
2. **CPU wall** — even if you had enough RAM, one CPU core processes rows sequentially. Processing 500 GB row-by-row takes hours.
3. **Disk I/O wall** — reading 500 GB from a single disk is a serial bottleneck.

**Distributed computing solves all three:**

| Problem | Single machine | Distributed (Spark) |
| --- | --- | --- |
| Memory | Must fit in RAM | Data split across 100s of machines |
| CPU | 1 core (or a few) | 1000s of cores working in parallel |
| Disk I/O | 1 disk reads serially | Many disks read simultaneously |

Spark's key insight: **split the data into partitions, spread them across a cluster, and process them in parallel.** You write simple code; Spark handles the distribution.

## 0.2 — Spark vs. Alternatives

| Tool | Best for | Limitations |
| --- | --- | --- |
| **pandas** | Small data (<10 GB), quick exploration, ML prototyping | Single machine only; crashes on large data |
| **SQL Warehouses** (Databricks SQL, BigQuery, Snowflake) | Interactive BI queries, dashboards, ad-hoc analytics | Limited to SQL; less flexible for ETL pipelines |
| **Apache Spark** | Large-scale ETL, ML pipelines, streaming, anything >10 GB | Overkill for small data; more setup complexity |

**When to use Spark:**
* Data too large for one machine
* Complex multi-step ETL pipelines
* Streaming + batch in one framework
* ML at scale (MLlib, distributed training)

**When pandas is fine:**
* Data fits comfortably in memory
* Quick ad-hoc analysis
* You need pandas-specific libraries (statsmodels, scikit-learn on small data)

## 0.3 — Lazy Evaluation (The Most Important Spark Concept)

Spark splits operations into two categories:

| Type | What it does | Runs immediately? | Examples |
| --- | --- | --- | --- |
| **Transformation** | Describes *what* to do — adds a step to the plan | No (lazy) | `filter`, `select`, `groupBy`, `join`, `withColumn` |
| **Action** | Demands a *result* — triggers execution of all pending transformations | Yes | `show`, `count`, `collect`, `write`, `first` |

**Why lazy?** Because it lets the **Catalyst Optimizer** see your *entire* plan before running anything. It can:
* Reorder operations (push filters before joins)
* Remove unused columns early
* Pick the best join strategy

**Mental model:** Think of transformations as writing a recipe. The action is saying "cook it now." Spark reads the full recipe, optimizes it, *then* executes.

```
df.filter(...)       # lazy — just adds a step to the recipe
  .select(...)       # lazy — another step
  .groupBy(...)      # lazy — another step
  .count()           # ACTION — now Spark optimizes and runs everything
```

In [0]:
# ============================================================
# HANDS-ON: Lazy Evaluation in Action
# ============================================================
# We build a small in-memory DataFrame (no files needed).
# Watch: transformations return INSTANTLY; actions take time.

data = [
    (1, 1, 12.50, "Toronto"),
    (2, 3, 40.00, "Toronto"),
    (3, 2,  8.75, "Vancouver"),
    (4, 1, 22.00, "Toronto"),
    (5, 4, 15.50, "Vancouver"),
]
columns = ["trip_id", "passengers", "fare", "city"]

df = spark.createDataFrame(data, columns)

# --- TRANSFORMATION (lazy: returns instantly, nothing computes) ---
big_trips = df.filter(df.fare > 10)

print("Transformation done — but NO data was processed yet!")
print(f"Type of big_trips: {type(big_trips)}")
print()

# --- ACTIONS (these actually trigger computation) ---
print("=" * 50)
print("ACTION: df.show() — triggers execution")
print("=" * 50)
df.show()

print(f"ACTION: df.count() = {df.count()}")
print()
print("ACTION: big_trips.show() — now the filter actually runs:")
big_trips.show()

## Phase 0 Checkpoint

**Can you explain this in one sentence?**
> Transformations (filter, select, groupBy) are lazy — they build a plan but do nothing. Only actions (show, count, collect, write) trigger actual computation.

**Can you answer these?**
1. Why can't pandas handle a 500 GB file? → It tries to load everything into RAM on one machine.
2. Is `df.filter(df.fare > 10)` a transformation or action? → Transformation (lazy, returns instantly).
3. Which triggers computation: `df.select("fare")` or `df.show()`? → `.show()` is the action.

---
---

# Phase 1 — Spark Architecture & the Compute Engine
**Duration:** Weeks 1–2 | **Goal:** Master the engine internals. This is your priority area.

## 1.1 — The Execution Hierarchy

Spark is a **distributed system** with four key components:

```
┌──────────────────────────────────────────────────┐
│              CLUSTER MANAGER                     │
│   (Databricks manages this for you)              │
│   Allocates resources, starts/stops executors     │
└──────────────────────────────────────────────────┘
         │                              │
┌──────────────────────┐     ┌──────────────────────┐
│       DRIVER             │     │     EXECUTORS          │
│  (The Brain)             │     │  (The Workers)         │
│                          │     │                        │
│  • Holds SparkSession    │     │  • Run tasks            │
│  • Builds the plan       │     │  • Hold data in memory  │
│  • Schedules work        │     │  • Report results back  │
│  • Collects results      │     │  • Multiple per cluster │
└──────────────────────┘     └──────────────────────┘
```

| Component | Role | Analogy |
| --- | --- | --- |
| **Driver** | Single node that holds your `SparkSession`, compiles your code into a plan, schedules tasks, and collects results | The project manager |
| **Executors** | Worker nodes that actually process data partitions in parallel | The team members doing the work |
| **Cluster Manager** | Allocates CPU/RAM to driver and executors (Databricks handles this) | HR — assigns people to teams |
| **Slots (cores)** | Each executor has multiple cores; one task runs per core | One desk per worker |

**Critical rule:** Never pull all your data to the **driver** (via `.collect()` or `.toPandas()` on large data). The driver is a single machine with limited RAM — it's meant to coordinate, not store data.

## 1.2 — How a Job Actually Runs

### Memorize This Chain:

```
Action  →  Job  →  Stages (split at shuffle boundaries)  →  Tasks (1 per partition)
```

| Term | Definition | Example |
| --- | --- | --- |
| **Job** | Triggered by ONE action | `df.count()` = 1 job; `df.show()` = 1 job |
| **Stage** | A set of tasks that can run without any shuffle in between | filter + select = 1 stage (no data movement) |
| **Task** | The smallest unit of work; processes ONE partition on ONE core | 8 partitions = 8 tasks running in parallel |
| **Shuffle** | Data redistribution across the network — **the #1 performance killer** | Happens during groupBy, join, distinct, repartition |

### How Stages Are Created

```
df.filter(fare > 10)     ───┐
  .select("city","fare")  ───┤  Stage 1 (narrow ops, no shuffle)
  .groupBy("city")        ───┘
       │
       │  ←←← SHUFFLE (data moves across network) ←←←  STAGE BOUNDARY
       │
  .count()               ───┐  Stage 2 (aggregate after shuffle)
  .show()                ───┘  (action triggers everything)
```

**Rule of thumb:** *N* stages = *N-1* shuffles. If you see 3 stages in the Spark UI, 2 shuffles happened.

## 1.3 — Partitions (The Atom of Spark)

A **partition** is a chunk of data. It determines your degree of parallelism:

| Partitions | Cores | Result |
| --- | --- | --- |
| 1 partition | 8 cores | 1 core busy, 7 idle — terrible! |
| 8 partitions | 8 cores | All 8 cores busy — perfect! |
| 1000 partitions | 8 cores | Works, but overhead from scheduling 1000 tiny tasks |

**Key config:** `spark.sql.shuffle.partitions` (default: **200**)
* Controls how many partitions are created *after* a shuffle (groupBy, join)
* For small data: 200 is way too many → lots of empty partitions
* AQE can auto-coalesce these at runtime (more on this in 1.4)

### Narrow vs. Wide Transformations

| Type | Shuffle? | Data movement | Examples |
| --- | --- | --- | --- |
| **Narrow** | No | Each output partition depends on ONE input partition | `filter`, `select`, `map`, `withColumn` |
| **Wide** | Yes | Output depends on ALL input partitions (data must move) | `groupBy`, `join`, `distinct`, `repartition`, `orderBy` |

**Wide transformations create a shuffle boundary (= new stage).** That’s why `groupBy` is expensive and `filter` is cheap.

In [0]:
# ============================================================
# HANDS-ON: Partitions & Parallelism
# ============================================================
from pyspark.sql.functions import spark_partition_id, count

# How many partitions does our DataFrame have?
num_partitions = df.select(spark_partition_id().alias("pid")).distinct().count()
print(f"Number of partitions in df: {num_partitions}")

# Row distribution across partitions
print("\nRows per partition:")
df.groupBy(spark_partition_id().alias("partition_id")) \
  .agg(count("*").alias("row_count")) \
  .orderBy("partition_id") \
  .show()

# What's the default shuffle partition count?
print(f"spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
print("\n→ This means after any groupBy/join, Spark creates 200 partitions by default.")
print("  For groupBy('city') with 2 distinct keys: only 2 partitions get data → 198 empty!")
print("  (It's about distinct KEY count, not row count. AQE will fix this.)")

In [0]:
# ============================================================
# NARROW transformation — NO shuffle
# ============================================================
# filter + select are narrow: each partition processes independently.
# Look at the plan: NO "Exchange" = no shuffle = one stage.

print("Physical plan for filter + select (NARROW):")
print("=" * 50)
df.filter(df.fare > 10).select("city", "fare").explain()
print()
print("→ Just 'LocalTableScan' — no Exchange, no shuffle, one stage.")
print("  Data stays where it is. Fast!")

In [0]:
# ============================================================
# WIDE transformation — SHUFFLE happens here
# ============================================================
# groupBy is wide: rows must move across partitions to group by city.
# Look at the plan: "Exchange" or "Shuffle" = data redistribution.

print("Physical plan for groupBy + count (WIDE):")
print("=" * 50)
df.groupBy("city").count().explain()
print()
print("→ You can see 'PhotonShuffleExchangeSink' (= Exchange = shuffle).")
print("  This creates a STAGE BOUNDARY. The job will have 2 stages.")
print()
print("Now run it and check the result:")
df.groupBy("city").count().show()

## 1.4 — The Engine Internals (Why Spark is So Fast)

You write simple Python. Behind the scenes, three engine layers optimize your code:

### 1. Catalyst Optimizer

**What:** Reads your *entire* lazy plan and rewrites it before executing.

**How it helps:**
* **Predicate pushdown** — moves filters as early as possible (less data to process)
* **Column pruning** — drops columns you never use
* **Join reordering** — picks the most efficient join order and strategy
* **Constant folding** — pre-computes static expressions

**This is why lazy evaluation exists.** Spark needs to see the full plan to optimize it.

### 2. Tungsten

**What:** Low-level engine that manages memory and generates optimized code.

**How it helps:**
* **Off-heap memory** — bypasses Java garbage collection overhead
* **Whole-stage code generation** — compiles your plan into tight Java bytecode (like hand-tuned C++)
* **Cache-friendly data layout** — stores data in binary format, not Java objects

**This is why DataFrames crush hand-written Python loops or RDD code.**

### 3. AQE (Adaptive Query Execution)

**What:** Re-optimizes *at runtime* using actual data sizes (not estimates).

**How it helps:**
* **Coalesces tiny partitions** — merges empty/small post-shuffle partitions (the demo below!)
* **Handles skew** — splits huge partitions that would bottleneck one task
* **Switches join strategy** — converts sort-merge join to broadcast join if one side is small

**On by default in Databricks.** Fixes many performance problems automatically.

---

### See the Optimizer with `.explain()`

The `.explain()` method shows the physical plan. Key things to spot:
* **Exchange** (or ShuffleExchange) = a shuffle = stage boundary
* **AdaptiveSparkPlan** = AQE is active
* **Photon\*** = Databricks’ native vectorized engine is handling it

In [0]:
# ============================================================
# AQE COALESCING DEMO
# ============================================================
# Default shuffle partitions = 200. Our data has only 2 cities.
# Without AQE: 200 partitions after shuffle (198 empty! wasteful).
# With AQE: Spark detects tiny partitions at runtime and merges them.

from pyspark.sql.functions import spark_partition_id, count

print("BEFORE the shuffle:")
print(f"  spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  → Spark will REQUEST 200 post-shuffle partitions\n")

# Run a wide operation (groupBy triggers shuffle)
result = df.groupBy("city").count()

# Force execution so AQE can finalize
result.collect()

# Check actual partitions in the result
actual = result.select(spark_partition_id().alias("pid")).distinct().count()

print("AFTER AQE optimization:")
print(f"  Requested partitions: 200")
print(f"  Actual partitions:    {actual}")
print(f"  → AQE coalesced 200 → {actual} because the data is tiny!")
print()
print("Show the plan (look for 'AdaptiveSparkPlan'):")
result.explain()

In [0]:
# ============================================================
# EXPERIMENT: Change shuffle partitions and observe
# ============================================================
# Reduce from 200 to 8 and see the difference

spark.conf.set("spark.sql.shuffle.partitions", "8")
print(f"Changed spark.sql.shuffle.partitions to: {spark.conf.get('spark.sql.shuffle.partitions')}")
print()

# Same groupBy, but now only 8 shuffle partitions requested
result_8 = df.groupBy("city").count()
result_8.collect()

actual_8 = result_8.select(spark_partition_id().alias("pid")).distinct().count()
print(f"With shuffle.partitions=8:  actual partitions = {actual_8}")
print(f"With shuffle.partitions=200: actual partitions = {actual} (from previous cell)")
print()
print("→ Fewer shuffle partitions = less overhead for small data.")
print("  For production big data, 200 (or auto via AQE) is usually fine.")

# Reset to default
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [0]:
# ============================================================
# JOIN DEMO — Wide transformation with TWO shuffles
# ============================================================
# Joins are WIDE: both sides may need to shuffle to align keys.
# Look for TWO "Exchange" nodes in the plan.

# Create a second DataFrame to join with
city_data = [
    ("Toronto", "Ontario", "Canada"),
    ("Vancouver", "BC", "Canada"),
]
city_columns = ["city", "province", "country"]
df_cities = spark.createDataFrame(city_data, city_columns)

# Join on city
joined = df.join(df_cities, on="city", how="inner")

print("Physical plan for JOIN (look for Exchange/Shuffle):")
print("=" * 50)
joined.explain()
print()
print("→ Join is WIDE: data from both sides must be co-located by key.")
print("  Spark may broadcast the small table (BroadcastExchange)")
print("  instead of shuffling both sides — that's the optimizer at work!")
print()
print("Join result:")
joined.show()

## Quick Reference — Key Concepts

| Question | Answer |
| --- | --- |
| Why can't pandas handle 500 GB? | Pandas loads everything into RAM on one machine. Spark distributes across many executors. |
| Is `df.filter(...)` a transformation or action? | **Transformation** — lazy, nothing computes. |
| Which triggers computation: `.select()` or `.show()`? | `.show()` — it's the action. |
| Driver vs. executor? | **Driver** = single coordinator. **Executors** = parallel workers. Never `.collect()` all data to the driver. |
| 3 stages → how many shuffles? | **2 shuffles.** Each shuffle = one stage boundary. |
| Is `join` narrow or wide? | **Wide** — rows must move across partitions. Shows `Exchange` in `.explain()`. |
| 1 partition on 8 cores → cores working? | **1 core.** One partition = one task = one core busy. |
| What does `spark.sql.shuffle.partitions` control? | Number of partitions created AFTER a shuffle (default 200). |
| What is AQE? | Adaptive Query Execution — re-optimizes at runtime, coalesces tiny partitions, fixes skew. |

### The Predict-Then-Verify Habit

1. Before running: call `.explain()` and predict where shuffles are (find `Exchange`)
2. Run the action
3. Open Spark UI → confirm stages match your prediction

This is the golden habit for performance tuning.

## Phase 1 Checkpoint

**Can you do these?**

- [ ] Given any query, predict where the shuffles are before running it
- [ ] Confirm your prediction via `.explain()` (look for Exchange)
- [ ] Explain the chain: Action → Job → Stages → Tasks
- [ ] Explain why `groupBy` is expensive but `filter` is cheap (wide vs narrow)
- [ ] Explain what Catalyst, Tungsten, and AQE each do in one sentence
- [ ] Know why you should never `.collect()` a large DataFrame to the driver

**One-sentence summaries (can you say these from memory?):**

1. **Catalyst:** Optimizes your lazy plan before execution — reorders, prunes, picks strategies.
2. **Tungsten:** Manages memory off-heap and generates tight bytecode — why DataFrames beat Python loops.
3. **AQE:** Re-optimizes at runtime with real data — coalesces partitions, fixes skew, switches joins.

---

**Next up: Phase 2 — DataFrames & Spark SQL (the API you’ll use 90% of the time)**